In [1]:
#Import any neccessary libraries 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error,median_absolute_error, mean_squared_error
import time


from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn import tree
import matplotlib.image as plting
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv("Nashville_housing_data_2013_2016.csv")

df.head()


,Unnamed: 0.1,Unnamed: 0,Parcel ID,Land Use,Property Address,Suite/ Condo #,Property City,Sale Date,Sale Price,Legal Reference,...,Building Value,Total Value,Finished Area,Foundation Type,Year Built,Exterior Wall,Grade,Bedrooms,Full Bath,Half Bath
0,0,0,105 03 0D 008.00,RESIDENTIAL CONDO,1208 3RD AVE S,8,NASHVILLE,2013-01-24,132000,20130128-0008725,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,105 11 0 080.00,SINGLE FAMILY,1802 STEWART PL,NaN,NASHVILLE,2013-01-11,191500,20130118-0006337,...,134400.0,168300.0,1149.00000,PT BSMT,1941.0,BRICK,C,2.0,1.0,0.0
2,2,2,118 03 0 130.00,SINGLE FAMILY,2761 ROSEDALE PL,NaN,NASHVILLE,2013-01-18,202000,20130124-0008033,...,157800.0,191800.0,2090.82495,SLAB,2000.0,BRICK/FRAME,C,3.0,2.0,1.0
3,3,3,119 01 0 479.00,SINGLE FAMILY,224 PEACHTREE ST,NaN,NASHVILLE,2013-01-18,32000,20130128-0008863,...,243700.0,268700.0,2145.60001,FULL BSMT,1948.0,BRICK/FRAME,B,4.0,2.0,0.0
4,4,4,119 05 0 186.00,SINGLE FAMILY,316 LUTIE ST,NaN,NASHVILLE,2013-01-23,102000,20130131-0009929,...,138100.0,164800.0,1969.00000,CRAWL,1910.0,FRAME,C,2.0,1.0,0.0


Part 1: Data Cleansing

In [2]:
#This cell checks if there are any missing datas and if so how many?

#Get rid of columnd of 0.1 and 0 that doesn't offer anything to the model
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

duplicate_count = df.duplicated().sum()
print(f"Number of exact duplicate rows: {duplicate_count}")
df = df.drop_duplicates().reset_index(drop=True)

print("Recognized Missing datas:")
print(df.isnull().sum())
print("\n")

print("Question Mark-missing data placeholders:")
print((df == "?").sum())

Number of exact duplicate rows: 103
Recognized Missing datas:
Parcel ID                                0
Land Use                                 0
Property Address                       159
Suite/ Condo   #                     50427
Property City                          159
Sale Date                                0
Sale Price                               0
Legal Reference                          0
Sold As Vacant                           0
Multiple Parcels Involved in Sale        0
Owner Name                           31318
Address                              30562
City                                 30562
State                                30562
Acreage                              30562
Tax District                         30562
Neighborhood                         30562
image                                31244
Land Value                           30562
Building Value                       30562
Total Value                          30562
Finished Area                      

In [3]:
#This is where we start cleaning the dataset up and split them up

# Standardize the column names by removing surrounding spaces,
# converting letters to lowercase, and replacing spaces with underscores.
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)




#Turns them into a binary yes and no for this column
df["multiple_parcels_involved_in_sale"] = (df["multiple_parcels_involved_in_sale"].str.strip().str.lower().map({
          "no": 0,
          "yes": 1
      })
)

#Turns them into a binary yes and no for this column also
df["sold_as_vacant"] = (df["sold_as_vacant"].str.strip().str.lower().map({
          "no": 0,
          "yes": 1
      })
)

# Convert the sale date from text into a datetime value.
df["sale_date"] = pd.to_datetime(
    df["sale_date"],
    errors="coerce"
)

# Extract useful time-related predictors from the original sale date.
df["sale_year"] = df["sale_date"].dt.year
df["sale_month"] = df["sale_date"].dt.month
df["sale_quarter"] = df["sale_date"].dt.quarter

#Get rid of predictors that would be unnecessary or redundant to the model
df = df.drop(columns=["parcel_id", "image", "state","legal_reference", "sale_date", "land_value",
                      "building_value","suite/_condo___#", "address","property_address","owner_name","city", "neighborhood",],errors="ignore")


# Identify all remaining text-based columns.
text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

# Standardize categorical values by removing excess internal and
# surrounding spaces
for column in text_columns:
    df[column] = (df[column].astype("string").str.replace(r"\s+", " ", regex=True).str.strip().replace("", pd.NA))

#Seperate the continuous target variable from predictors variable 
y = df["sale_price"]
x = df.drop(columns=["sale_price"])

# Split the data before replacing missing values to prevent information
# from the testing data from leaking into the training process.
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.30,random_state=42)

#Identifies columns that are numerical
numerical_columns = x_train.select_dtypes(
    include="number"
).columns.tolist()

#print(numerical_columns)

#Calculates the median using training data to replace missing data in the two sets for numerical columns
for column in numerical_columns:
    training_median = x_train[column].median()
    x_train[column] = x_train[column].fillna(training_median)
    x_test[column] = x_test[column].fillna(training_median)
    
#Identify all of the categorical columns
categorical_columns = x_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

#print(categorical_columns)

#Get the most frequently occuring property_city as it is missing a small amount of data
property_city_mode = x_train["property_city"].mode()[0]

# replaces the data that are not property city with unknown as they have a lot of missing datas
for column in categorical_columns:
    #if the column is property_city, replace those recognized missing value
    if column == "property_city":
        x_train[column] = x_train[column].fillna(property_city_mode)

        x_test[column] = x_test[column].fillna(property_city_mode)
    else:
        x_train[column] = x_train[column].fillna("Unknown")
        x_test[column] = x_test[column].fillna("Unknown")


# Confirm that all missing predictor values were handled successfully.
print("Training missing values:",x_train.isna().sum().sum())

print("Testing missing values:",x_test.isna().sum().sum())

Training missing values: 0
Testing missing values: 0


In [4]:
#This cell sets the categorical columns and numerical columns up straight with get_dummies and standard scaling

#Does OneHotEncoding on the Categorical Columns
x_train_encoded = pd.get_dummies(x_train,columns=categorical_columns,drop_first=True,dtype=int)
x_test_encoded = pd.get_dummies(x_test,columns=categorical_columns,drop_first=True,dtype=int)

#Rearranges the test columns to match train columns order
x_test_encoded = x_test_encoded.reindex(columns=x_train_encoded.columns,fill_value=0)

#The columns made up of binary values, which are 0 and 1
binary_columns = ["sold_as_vacant","multiple_parcels_involved_in_sale"]

#Identifies the non-binary columns
continuous_columns = [column for column in numerical_columns
                      if column not in binary_columns]

scaler = StandardScaler()

# Learn the means and standard deviations from training data
x_train_encoded[continuous_columns] = scaler.fit_transform(
    x_train_encoded[continuous_columns]
)

# Apply the same training transformation to testing data
x_test_encoded[continuous_columns] = scaler.transform(
    x_test_encoded[continuous_columns]
)

#Checks the shape of the predictors of training and test
print("Training shape:", x_train_encoded.shape)
print("Testing shape:", x_test_encoded.shape)

#Checks to see if the columns are equal to each other
print(
    "Columns match:",
    x_train_encoded.columns.equals(
        x_test_encoded.columns
    )
)

Training shape: (39573, 99)
Testing shape: (16960, 99)
Columns match: True


Part 2: Building the linear regression and see which variable is driving those prices

In [5]:
#This cell creates the OLS  linear regression  for the training data

x2 = sm.add_constant(x_train_encoded) #add the constant to the training set

est = sm.OLS(y_train,x2).fit()

print(est.summary())

                            OLS Regression Results                            
Dep. Variable:             sale_price   R-squared:                       0.135
Model:                            OLS   Adj. R-squared:                  0.133
Method:                 Least Squares   F-statistic:                     66.15
Date:                Sun, 02 Aug 2026   Prob (F-statistic):               0.00
Time:                        23:03:57   Log-Likelihood:            -5.9802e+05
No. Observations:               39573   AIC:                         1.196e+06
Df Residuals:                   39479   BIC:                         1.197e+06
Df Model:                          93                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [6]:
#Now test the model on the test data

x2test = sm.add_constant(x_test_encoded) #adds the constant to the test predictors

y_pred = est.predict(x2test)

test_r2 = r2_score(y_test,y_pred)

print("For the test data: ")
print(f"R^2 = {test_r2:.4f}")



For the test data: 
R^2 = 0.1195


In [7]:
# Identify statistically significant predictors from the full model


#Set the statistical significance threshold to 0.05, where predictors with p-values below
#0.05 will be considered significant
significance_level = 0.05 

#Creates a dataframe that contains each predictor's coefficient and p-value from the
#linear regression model
significance_results = pd.DataFrame({
    "Variable": est.params.index,
    "Coefficient": est.params.values,
    "P_Value": est.pvalues.values
})

# Remove the constant and retain variables with p-values below 0.05
significant_results = (
    significance_results[
        (significance_results["Variable"] != "const") &
        (significance_results["P_Value"] < significance_level)
    ]
    .sort_values(by="P_Value")
    .reset_index(drop=True)
)

#Display the statistically significant predictors along with their coefficient and p-value
print("Statistically significant predictors:")
print(significant_results)

#Displays the total number of significant predictors 
significant_variables = significant_results["Variable"].tolist()
print("\nNumber of significant predictors:",
      len(significant_variables))

Statistically significant predictors:
                                Variable   Coefficient       P_Value
0      multiple_parcels_involved_in_sale  1.191210e+06  0.000000e+00
1                         sold_as_vacant -4.379487e+05  3.268421e-61
2                            total_value  1.779089e+05  1.333290e-48
3                property_city_NASHVILLE  1.119929e+05  6.119822e-13
4                              sale_year  2.976363e+04  2.617651e-10
5                             sale_month -1.016957e+05  1.006062e-09
6                           sale_quarter  9.094331e+04  4.563795e-08
7                  property_city_MADISON -1.260714e+05  2.177260e-05
8                              grade_AAB  2.140717e+06  2.455720e-03
9                property_city_BRENTWOOD  8.775763e+04  2.828787e-03
10  land_use_GREENBELT/RES GRRENBELT/RES -2.954669e+06  2.991211e-02
11            land_use_MORTUARY/CEMETERY -3.199456e+06  3.768884e-02
12                               acreage  1.029176e+04  4.036481e

Part 3: Decision Tree

In [8]:

#Creates a Decision Tree Regression model with a max depth of 10.
#Limiting the depth would prevent the tree from overfitting.
#random_state=0 ensures the reproducible results
treemodel = DecisionTreeRegressor(max_depth=10)

#Trains the decision tree with the training datasets
model = treemodel.fit(x_train_encoded,y_train)

#Predicts the sale prices to see how well the model fits and accurately predicts the price sale
tree_train_predictions = model.predict(x_train_encoded)
tree_test_predictions = model.predict(x_test_encoded)


#Calulates the training and test R^2 and prints them out
tree_train_r2 = r2_score(y_train, tree_train_predictions)
tree_test_r2 = r2_score(y_test, tree_test_predictions)

print(f"Training R^2: {tree_train_r2:.3f}")
print(f"Testing R^2:  {tree_test_r2:.3f}")


Training R^2: 0.474
Testing R^2:  0.471


In [9]:
# Determine which predictors are most important to the decision tree

tree_importance = pd.DataFrame({
    "Predictor": x_train_encoded.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(tree_importance.head(5))

                           Predictor  Importance
0                         sale_month    0.342000
1         land_use_RESIDENTIAL CONDO    0.203726
2                          sale_year    0.132695
3  multiple_parcels_involved_in_sale    0.125763
4                        total_value    0.075140


Part 4: Random Forest

In [10]:

#Creates the Random Forest regression model using 40 decision trees.
#random_state=0 ensures that the model reproduces the same results
rfclass = RandomForestRegressor(n_estimators = 40, random_state=0)

#Trains the random forest using the encoded training predictors and their corresponding sale prices
rfclass.fit(x_train_encoded,y_train)

#Calculates the training and test R^2 to see how close the random forest fits the data it was trained on
# and how well it predicts the sale prices
print("Random Forest Train R^2: ", rfclass.score(x_train_encoded,y_train))
print("Random Forest Test R^2: ", rfclass.score(x_test_encoded,y_test))



Random Forest Train R^2:  0.5000911779397539
Random Forest Test R^2:  0.49847163964439256


In [11]:
#Rank the decision tree's predictors by importance to see which drives the model from highest to lowest

rf_predictors = pd.DataFrame({
    "Predictor": x_train_encoded.columns,
    "Importance": rfclass.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(rf_predictors.head(5))

                           Predictor  Importance
0                         sale_month    0.331326
1         land_use_RESIDENTIAL CONDO    0.195982
2                          sale_year    0.142913
3  multiple_parcels_involved_in_sale    0.114656
4                        total_value    0.072372


Part 5: Gradient Boost

In [12]:
#Creates the Gradient Boosting regression model using 40 sequential trees
#random_state ensures that the results would be reproduced
gbclass = GradientBoostingRegressor(n_estimators = 40, random_state=0)

#Trains the model using the encoded training predictors and sale price
gbclass.fit(x_train_encoded,y_train)


#Calculates the training and test R^2 to see how well the model fits the data
#and how well it predicts the sale prices
print("Training Gradient Boost R^2:",gbclass.score(x_train_encoded, y_train))
print("Testing Gradient Boost R^2:",gbclass.score(x_test_encoded, y_test))

Training Gradient Boost R^2: 0.3295901548449507
Testing Gradient Boost R^2: 0.3122404640737677


In [13]:
#It ranks the Gradient Boost model's predictors by importance to see which drives the model from highest to lowest

#Puts them in a dataframe to be printed out
Gradient_predictors = pd.DataFrame({
    "Predictor": x_train_encoded.columns,
    "Importance": gbclass.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

print(Gradient_predictors.head(5))

                           Predictor  Importance
0         land_use_RESIDENTIAL CONDO    0.289376
1                         sale_month    0.257769
2  multiple_parcels_involved_in_sale    0.197989
3                        total_value    0.095725
4                     sold_as_vacant    0.059880


Part 6: Benchmarking models between Decision Tree, Random Forest, and Gradient Boost

In [14]:
#This cell is where the models are benchmarked aganist each other using the same training and testing datasets

benchmark_models= { "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=0),
                    "Random Forest": RandomForestRegressor(n_estimators=40,random_state=0),
                    "Gradient Boosting": GradientBoostingRegressor(n_estimators = 40, random_state=0)
                  }

#Create a list for storing each model's benchmarking results
benchmark = []
#Creates a dictionary for storing the fitted models.
fitted_models = {}

#Train and evaluate each regression model.
for model_name, regression_model in benchmark_models.items():

    #start the timer
    computation_start = time.perf_counter()

    #Trains the model
    regression_model.fit(x_train_encoded,y_train)

    #predict the testing prices
    test_predictions = regression_model.predict(x_test_encoded)

    #Calculate the total time for model to process
    total_compute_time =  (time.perf_counter() - computation_start)

    #Predict the training prices to see how the model fits the data it was trained on.
    train_predictions = regression_model.predict(x_train_encoded)

    #Save the fitted model for possible later analysis
    fitted_models[model_name] = regression_model

    #Calculate and store the benchmarking metrics.
    benchmark.append({
        "Model": model_name,

        #Measures how well the model fits the training data
        "Training R^2": r2_score(y_train, train_predictions),

        #Measures the accuracy of the model on test datasets
        "Testing R^2": r2_score(y_test, test_predictions),

        #calculate the average absolute testing error
        "R^2 Gap": r2_score(y_train, train_predictions) - r2_score(y_test, test_predictions) ,

        # Calculates RMSE, which gives greater weight to large errors.
        "Testing RMSE": np.sqrt(mean_squared_error(y_test, test_predictions)),

        #Stores the training and testing-prediction computation time
        "Computation Time": total_compute_time
    })

#Converts the stored results into a DataFrame and rank the models from highest to lowest
# testing R^2
benchmark_results = (
    pd.DataFrame(benchmark).sort_values(by="Testing R^2",ascending=False).reset_index(drop=True)
)

display(
    benchmark_results.style.format({
        "Training R^2": "{:.3f}",
        "Testing R^2": "{:.3f}",
        "R^2 Gap": "{:.3f}",
        "Testing RMSE": "${:.2f}",
        "Computation Time": "{:.4f} seconds"
    })
)

,Model,Training R^2,Testing R^2,R^2 Gap,Testing RMSE,Computation Time
0,Random Forest,0.500,0.498,0.002,$621501.03,10.3436 seconds
1,Decision Tree,0.474,0.471,0.003,$638319.15,0.2012 seconds
2,Gradient Boosting,0.330,0.312,0.017,$727800.61,4.0289 seconds
